In [ ]:
import ROOT
from ROOT import TVirtualFitter
from ROOT import TMath
filePath = '/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k60100/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root'
file = ROOT.TFile(filePath)

fitterList, objList, canvasList = [], [], []
# print(f"Available objects in file: {file.GetListOfKeys()}")
objPath = 'PtCandBin_25_30/PtHadBin_2_30/DeltaPhiBin_-1570_-1178/hCorrectedCorrel'
objList.append(file.Get(objPath))
obj = objList[-1]
obj.SetDirectory(0)
# obj.Rebin(2)
obj.SetStats(0)

# ROOT.Math.MinimizerOptions.SetDefaultMinimizer("Minuit2")
# ROOT.Math.MinimizerOptions.SetDefaultMaxFunctionCalls(10000)

fitterList.append(TVirtualFitter)
fitter = fitterList[-1]
fitter.SetMaxIterations(50000)
fitter.SetDefaultFitter("Minuit2")

# $f(\Delta\phi) = \text{Flow}(\Delta\phi) + A_{\text{near}} e^{-\frac{\Delta\phi^2}{2\sigma_{\text{near}}^2}} + A_{\text{away}} e^{-\frac{(\Delta\phi - \pi)^2}{2\sigma_{\text{away}}^2}}$

# fitFunc = ROOT.TF1("fitFunc", "[0] * TMath::Exp(-TMath::Power(x, 2) / (2 * TMath::Power([1], 2))) + [2] * TMath::Exp(-TMath::Power(x - TMath::Pi(), 2) / (2 * TMath::Power([3], 2))) + [4]", -TMath.Pi()/2, 3 * TMath.Pi()/2)

# fitFunc = ROOT.TF1("fitFuncVM", 
#     "[0] * TMath::Exp( [1] * (TMath::Cos(x) - 1) ) + "
#     "[2] * TMath::Exp( [3] * (TMath::Cos(x - TMath::Pi()) - 1) ) + "
#     "[4]", 
#     -TMath.Pi()/2, 3*TMath.Pi()/2)

# fit_formula = (
#     "[0] + "
#     "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-TMath::Power(x, 2) / (2 * TMath::Power([2], 2))) + "
#     "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-TMath::Power(x - TMath::Pi(), 2) / (2 * TMath::Power([4], 2)))"
# )
# fitFunc = ROOT.TF1("fitFunc", fit_formula, -TMath.Pi()/2, 3 * TMath.Pi()/2)

fit_formula = (
    "[0] + "
    "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(- x                  *  x                  / (2*[2]*[2])) + "
    "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x - 2*TMath::Pi()) * (x - 2*TMath::Pi()) / (2*[2]*[2])) + "
    # "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x + 2*TMath::Pi()) * (x + 2*TMath::Pi()) / (2*[2]*[2])) + "
    "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-(x - TMath::Pi())   * (x - TMath::Pi())   / (2*[4]*[4])) + "
    # "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-(x - 3*TMath::Pi()) * (x - 3*TMath::Pi()) / (2*[4]*[4])) + "
    "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-(x + TMath::Pi())   * (x + TMath::Pi())   / (2*[4]*[4]))"
)
fitFunc = ROOT.TF1("fitFunc", fit_formula, -TMath.Pi()/2, 3*TMath.Pi()/2)

minBin = obj.GetXaxis().FindBin(obj.GetMinimum())
maxBin = obj.GetXaxis().FindBin(obj.GetMaximum())
lowestErr = obj.GetBinError(minBin)
highestErr = obj.GetBinError(maxBin)

fitFunc.SetParameter(0, obj.GetMinimum())
fitFunc.SetParLimits(0, obj.GetMinimum()-lowestErr, obj.GetMaximum()+highestErr)
fitFunc.SetParameters(1, 0.5*(obj.GetMaximum() - obj.GetMinimum()))
fitFunc.SetParameter(2, 0.5)
# fitFunc.SetParLimits(2, 0.0001, 2*TMath.Pi())
fitFunc.SetParameter(3, 0.5*(obj.GetMaximum() - obj.GetMinimum()))
fitFunc.SetParameter(4, 0.5)
# fitFunc.SetParLimits(4, 0.0001, 2*TMath.Pi())
fitFunc.SetParNames("Baseline", "A Near", "Sigma Near", "A Away", "Sigma Away")


fitFunc.SetLineColor(ROOT.kRed)
fitFunc.SetLineWidth(2)
# obj.Fit(fitFunc, "R")
fitRes = obj.Fit(fitFunc, "RIS")

if int(fitRes) == 0:
    print("Fit Successful!")
    
    correlMatrix = fitRes.GetCorrelationMatrix()
    print("\nCorrelation Matrix:")
    n_pars = fitFunc.GetNpar()
    for i in range(n_pars):
        row = []
        for j in range(n_pars):
            row.append(f"{correlMatrix(i, j):.4f}")
        print("\t".join(row))

    covMatrix = fitRes.GetCovarianceMatrix()
    print("\nCovariance Matrix:")
    for i in range(n_pars):
        row = []
        for j in range(n_pars):
            row.append(f"{covMatrix(i, j):.4e}")
        print("\t".join(row))
else:
    print("Fit Failed. Matrix extraction skipped.")
    print(f"Fit Status Code: {int(fitRes)}")


textPad = ROOT.TPaveText(0.15, 0.56, 0.45, 0.90, "NDC")
textPad.SetFillStyle(0)
textPad.SetBorderSize(0)
textPad.SetTextAlign(12)
textPad.SetTextSize(0.03)
textPad.AddText(f"Baseline: {fitFunc.GetParameter(0):.3f} \\pm {fitFunc.GetParError(0):.3f}")
textPad.AddText(f"A Near: {fitFunc.GetParameter(1):.3f} \\pm {fitFunc.GetParError(1):.3f}")
textPad.AddText(f"Sigma Near: {fitFunc.GetParameter(2):.3f} \\pm {fitFunc.GetParError(2):.3f}")
textPad.AddText(f"A Away: {fitFunc.GetParameter(3):.3f} \\pm {fitFunc.GetParError(3):.3f}")
textPad.AddText(f"Sigma Away: {fitFunc.GetParameter(4):.3f} \\pm {fitFunc.GetParError(4):.3f}")
textPad.AddText(f"Chi2/NDF: {fitRes.Chi2()/fitRes.Ndf():.2f}")

tfBase = ROOT.TF1("tfBase", "[0]", -TMath.Pi()/2, 3*TMath.Pi()/2)
tfBase.SetParameter(0, fitFunc.GetParameter(0))
tfBase.SetLineColor(ROOT.kBlue)
tfBase.SetLineWidth(2)
canvasList.append(ROOT.TCanvas("canvas", "Fit Result", 800, 600))
canvas = canvasList[-1]
obj.Draw("E")
fitFunc.Draw("Same")
tfBase.Draw("Same")
textPad.Draw()
canvas.SaveAs("fit_result_woper.png")




In [ ]:
import ROOT
from ROOT import TVirtualFitter
from ROOT import TMath
import os
def perform_fit(histo, outPath="./", outName="fit_result.png", fitFunction='Gaus'):
    ROOT.Math.MinimizerOptions.SetDefaultMinimizer("Minuit2")
    ROOT.Math.MinimizerOptions.SetDefaultMaxFunctionCalls(50000)
    
    if fitFunction == 'Gaus':
        fit_formula = (
            "[0] + "
            "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-TMath::Power(x, 2) / (2 * TMath::Power([2], 2))) + "
            "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-TMath::Power(x - TMath::Pi(), 2) / (2 * TMath::Power([4], 2)))"
        )
    elif fitFunction == 'GausPeriodic':
        fit_formula = (
            "[0] + "
            "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(- x                  *  x                  / (2*[2]*[2])) + "
            "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x - 2*TMath::Pi()) * (x - 2*TMath::Pi()) / (2*[2]*[2])) + "
            "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x + 2*TMath::Pi()) * (x + 2*TMath::Pi()) / (2*[2]*[2])) + "
            "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-(x - TMath::Pi())   * (x - TMath::Pi())   / (2*[4]*[4])) + "
            "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-(x - 3*TMath::Pi()) * (x - 3*TMath::Pi()) / (2*[4]*[4])) + "
            "[3]/(TMath::Sqrt(2*TMath::Pi()) * [4]) * TMath::Exp(-(x + TMath::Pi())   * (x + TMath::Pi())   / (2*[4]*[4]))"
        )
    else:
        raise ValueError(f"Unsupported fit function: {fitFunction}")

    fitFunc = ROOT.TF1("fitFunc", fit_formula, -TMath.Pi()/2, 3*TMath.Pi()/2)

    minBin = histo.GetXaxis().FindBin(histo.GetMinimum())
    maxBin = histo.GetXaxis().FindBin(histo.GetMaximum())
    lowestErr = histo.GetBinError(minBin)
    highestErr = histo.GetBinError(maxBin)
    binWidth = histo.GetXaxis().GetBinWidth(1)

    # minusHalfPiBin = histo.GetXaxis().FindBin(-TMath.Pi()/2)
    # halfPiBin = histo.GetXaxis().FindBin(TMath.Pi()/2)
    # thirdHalfPiBin = histo.GetXaxis().FindBin(3*TMath.Pi()/2)
    # yNear = (histo.Integral(minusHalfPiBin, halfPiBin) - histo.GetMinimum()) / histo.GetNbinsX() / 2 / binWidth
    # yAway = (histo.Integral(halfPiBin, thirdHalfPiBin) - histo.GetMinimum()) / histo.GetNbinsX() / 2 / binWidth

    fitFunc.SetParameter(0, histo.GetMinimum()+lowestErr)
    fitFunc.SetParLimits(0, histo.GetMinimum(), histo.GetMaximum()+highestErr)

    fitFunc.SetParameters(1, 0.5*(histo.GetMaximum() - histo.GetMinimum()))
    fitFunc.SetParLimits(1, 0.01, histo.GetMaximum() - histo.GetMinimum())

    fitFunc.SetParameter(2, TMath.Pi()/8)
    fitFunc.SetParLimits(2, TMath.Pi()/16, TMath.Pi())

    fitFunc.SetParameter(3, 0.5*(histo.GetMaximum() - histo.GetMinimum()))
    fitFunc.SetParLimits(3, 0.01, histo.GetMaximum() - histo.GetMinimum())

    fitFunc.SetParameter(4, TMath.Pi()/8)
    fitFunc.SetParLimits(4, TMath.Pi()/16, TMath.Pi())
    
    fitFunc.SetParNames("Baseline", "A Near", "Sigma Near", "A Away", "Sigma Away")
    
    fitFunc.SetLineColor(ROOT.kRed)
    fitFunc.SetLineWidth(4)
    # histo.Fit(fitFunc, "R")
    fitRes = histo.Fit(fitFunc, "RIS")
    
    
    if int(fitRes) == 0:
        print("Fit Successful!")
        
        correlMatrix = fitRes.GetCorrelationMatrix()
        print("\nCorrelation Matrix:")
        n_pars = fitFunc.GetNpar()
        for i in range(n_pars):
            row = []
            for j in range(n_pars):
                row.append(f"{correlMatrix(i, j):.4f}")
            print("\t".join(row))

        covMatrix = fitRes.GetCovarianceMatrix()
        print("\nCovariance Matrix:")
        for i in range(n_pars):
            row = []
            for j in range(n_pars):
                row.append(f"{covMatrix(i, j):.4e}")
            print("\t".join(row))
    else:
        print("Fit Failed. Matrix extraction skipped.")
        print(f"Fit Status Code: {int(fitRes)}")
        
    textPad = ROOT.TPaveText(0.15, 0.70, 0.45, 0.95, "NDC")
    textPad.SetFillStyle(0)
    textPad.SetBorderSize(0)
    textPad.SetTextAlign(12)
    textPad.SetTextFont(42)
    textPad.SetTextSize(0.03)
    textPad.AddText(f"Baseline: {fitFunc.GetParameter(0):.3f} \\pm {fitFunc.GetParError(0):.3f}")
    textPad.AddText(f"A Near: {fitFunc.GetParameter(1):.3f} \\pm {fitFunc.GetParError(1):.3f}")
    textPad.AddText(f"\\sigma\\ Near: {fitFunc.GetParameter(2):.3f} \\pm {fitFunc.GetParError(2):.3f}")
    textPad.AddText(f"A Away: {fitFunc.GetParameter(3):.3f} \\pm {fitFunc.GetParError(3):.3f}")
    textPad.AddText(f"\\sigma\\ Away: {fitFunc.GetParameter(4):.3f} \\pm {fitFunc.GetParError(4):.3f}")
    textPad.AddText(f"Chi2/NDF: {fitRes.Chi2()/fitRes.Ndf():.2f}")
    
    leg = ROOT.TLegend(0.80, 0.85, 0.95, 0.95)
    leg.SetFillStyle(0)
    leg.SetBorderSize(0)
    leg.SetTextSize(0.02)
    leg.AddEntry(fitFunc, "Total Fit", "L")
    
    tfBase = ROOT.TF1("tfBase", "[0]", -TMath.Pi()/2, 3*TMath.Pi()/2)
    tfBase.SetParameter(0, fitFunc.GetParameter(0))
    tfBase.SetLineColor(ROOT.kGray+2)
    tfBase.SetLineWidth(3)
    leg.AddEntry(tfBase, "Baseline", "L")
    
    near_formula = "[0] + [1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-x*x / (2*[2]*[2]))"
    if fitFunction == 'GausPeriodic':
        near_formula += " + [1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x - 2*TMath::Pi()) * (x - 2*TMath::Pi()) / (2*[2]*[2])) + "
        near_formula += "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x + 2*TMath::Pi()) * (x + 2*TMath::Pi()) / (2*[2]*[2]))"
    fSubNear = ROOT.TF1("fSubNear", near_formula, -TMath.Pi()/2, 3*TMath.Pi()/2)
    # fSubNear.SetParameters(fitFunc.GetParameter(0), fitFunc.GetParameter(1), fitFunc.GetParameter(2))
    fSubNear.SetParameter(0, fitFunc.GetParameter(0))
    fSubNear.SetParameter(1, fitFunc.GetParameter(1))
    fSubNear.SetParameter(2, fitFunc.GetParameter(2))
    fSubNear.SetLineColor(ROOT.kAzure+2)
    fSubNear.SetLineWidth(3)
    fSubNear.SetLineStyle(9)
    leg.AddEntry(fSubNear, "Near-Side Gaus.", "L")

    away_formula = "[0] + [1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x-TMath::Pi())*(x-TMath::Pi())/(2*[2]*[2]))"
    if fitFunction == 'GausPeriodic':
        away_formula += " + [1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x-3*TMath::Pi())*(x-3*TMath::Pi())/(2*[2]*[2])) + "
        away_formula += "[1]/(TMath::Sqrt(2*TMath::Pi()) * [2]) * TMath::Exp(-(x+TMath::Pi())*(x+TMath::Pi())/(2*[2]*[2]))"
    fSubAway = ROOT.TF1("fSubAway", away_formula, -TMath.Pi()/2, 3*TMath.Pi()/2)
    fSubAway.SetParameter(0, fitFunc.GetParameter(0))
    fSubAway.SetParameter(1, fitFunc.GetParameter(3))
    fSubAway.SetParameter(2, fitFunc.GetParameter(4))
    fSubAway.SetLineColor(ROOT.kGreen+2)
    fSubAway.SetLineWidth(3)
    fSubAway.SetLineStyle(9)
    leg.AddEntry(fSubAway, "Away-Side Gaus.", "L")
    # histo.GetListOfFunctions().Add(fitFunc)
    # histo.GetListOfFunctions().Add(tfBase)
    # histo.GetListOfFunctions().Add(fSubNear)
    # histo.GetListOfFunctions().Add(fSubAway)
    # fLM->SetParameter(0, par[0]);
    # fLM->SetParameter(1, par[1]);
    # fLM->SetLineColor(kAzure+2);
    # fLM->SetLineStyle(8); // dashed
    # fLM->SetLineWidth(4);

    # // Flow (total flow component: G * [1 + 2*v2*cos(2x) + 2*v3*cos(3x)])
    # fFlow->SetParameter(0, par[1]);  // g
    # fFlow->SetParameter(1, par[2]); // v2
    # fFlow->SetParameter(2, par[3]); // v3
    # fFlow->SetLineColor(kGreen+2);
    # fFlow->SetLineStyle(9); // dotted
    # fFlow->SetLineWidth(4);

    canvas = ROOT.TCanvas(f"canvas_{objkey.replace('/', '_')}", "Fit Result", 1600, 1200)
    canvas.SetLeftMargin(0.15)
    canvas.SetRightMargin(0.05)
    canvas.SetBottomMargin(0.12)
    canvas.SetTopMargin(0.05)

    histo.SetStats(0)
    histo.Draw("E")
    fitFunc.DrawClone("Same")
    tfBase.DrawClone("Same")
    fSubNear.DrawClone("Same")
    fSubAway.DrawClone("Same")
    textPad.Draw()
    leg.Draw()

    os.makedirs(outPath, exist_ok=True)
    canvas.Modified()
    canvas.Update()
    canvas.SaveAs(f"{outPath}/{outName}")
    
    outRootName = outName.replace(".png", ".root")
    outFile = ROOT.TFile(f"{outPath}/{outRootName}", "RECREATE")
    outFile.cd()
    canvas.Write("fitCanvas")
    
    # fitFunc.SetName("totFitFunc")
    # fitFunc.Write("totFitFunc")
    
    # tfBase.SetName("baselineFunction")
    # tfBase.Write("baselineFunction")
    
    # fSubNear.SetName("nearSideFunction")
    # fSubNear.Write("nearSideFunction")
    
    # fSubAway.SetName("awaySideFunction")
    # fSubAway.Write("awaySideFunction")

    histo.SetName("dataHistogram")
    histo.Write("dataHistogram")

    outFile.Close()
    




In [ ]:
filePath = '/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k60100/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root'
file = ROOT.TFile(filePath)

def find_specific_objects(directory, target_class=None, name_pattern=None, current_path=""):
    found_objects = {}
    
    for key in directory.GetListOfKeys():
        obj_name = key.GetName()
        obj_class = key.GetClassName()
        full_path = f"{current_path}/{obj_name}"
        
        if obj_class == "TDirectoryFile" or obj_class == "TDirectory":
            sub_dir = key.ReadObj()
            found_objects.update(find_specific_objects(sub_dir, target_class, name_pattern, full_path))
            continue
            
        match_class = (target_class is None) or (obj_class == target_class)
        match_name = (name_pattern is None) or (name_pattern in full_path)
        
        if match_class and match_name:
            print(f"Found {obj_class} at {full_path}")
            obj = key.ReadObj()
            obj.SetDirectory(0)  # Detach from file to prevent it from being deleted
            found_objects[full_path] = obj

    return found_objects

objList = find_specific_objects(file, target_class="TH1D", name_pattern="DeltaPhiBin_-1570_-1178/hCorrectedCorrel")
for objkey, obj in objList.items():
    obj.Rebin(2)
    perform_fit(obj, outPath="./rebin2", outName=f"fit{objkey.replace('/', '_')}.png", fitFunction='GausPeriodic')


In [ ]:
import os
import ROOT
import numpy as np


path = "/home/wuct/ALICE/reps/cfAnRes/tools/notebook/2pc/smooth"
outPath = f"{path}/comparison"
os.makedirs(outPath, exist_ok=True)

# 1. 严格对齐文件
subdirs = ['raw', 'rebin2', 'rebin4']
root_files = {d: [] for d in subdirs}

for d in subdirs:
    full_dir = os.path.join(path, d)
    if os.path.exists(full_dir):
        for file in os.listdir(full_dir):
            if file.endswith(".root"):
                root_files[d].append(os.path.join(full_dir, file))
        root_files[d].sort()

num_plots = min(len(root_files['raw']), len(root_files['rebin2']), len(root_files['rebin4']))
print(f"Aligned bins to compare: {num_plots}")

# 用来存放采样后纯离散点直方图的字典
graphs = {d: [] for d in subdirs}

# 采样配置：在 [-pi/2, 3*pi/2] 之间均匀采样 500 个点，连成平滑曲线
x_samples = np.linspace(-ROOT.TMath.Pi()/2, 3*ROOT.TMath.Pi()/2, 500)

# 2. 读取与离散化采样循环 (不解析任何公式，绝不死锁)
print("Executing fast text-free sampling loop...")
for subdir in subdirs:
    for idx, file in enumerate(root_files[subdir][:num_plots]):
        root_file = ROOT.TFile.Open(file, "READ")
        if not root_file or root_file.IsZombie():
            continue
            
        dataHistogram = root_file.Get("dataHistogram")
        if not dataHistogram:
            print(f"dataHistogram missing in {file}")
            root_file.Close()
            continue
            
        # 寻找拟合函数
        fit_func = None
        for func in dataHistogram.GetListOfFunctions():
            if "fitFunc" in func.GetName() or isinstance(func, ROOT.TF1):
                fit_func = func
                break
                
        if fit_func:
            # 【黑科技核心】：创建一个纯数据的 TH1D，用来存放采样点
            h_sample = ROOT.TH1D(f"h_sample_{subdir}_{idx}", "", len(x_samples)-1, -ROOT.TMath.Pi()/2, 3*ROOT.TMath.Pi()/2)
            h_sample.SetDirectory(0) # 彻底断绝与文件的关系，闭眼求值
            
            for bin_idx, x_val in enumerate(x_samples):
                # 直接通过 C++ 编译好的二进制内存直接求值 (Eval)，不经过任何 TFormula 解析
                y_val = fit_func.Eval(x_val)
                h_sample.SetBinContent(bin_idx + 1, y_val)
                
            graphs[subdir].append(h_sample)
        else:
            print(f"Warning: No fit function found in {file}")
            
        root_file.Close() # 读完当场直接关闭文件，内存极其安全干净！

print("Sampling completed successfully without any Cling block.")

# 3. 画图对比循环
print("\nStarting Plotting Loop...")
canvasList = []
legendList = []

for iPt in range(num_plots):
    canvas = ROOT.TCanvas(f"canvas_{iPt}", f"Fit Comparison Bin_{iPt}", 1600, 1200)
    canvasList.append(canvas)
    
    # 样式配置（此时它们全是纯数据的普通直方图线段了，非常听话）
    graphs['raw'][iPt].SetLineColor(ROOT.kRed)
    graphs['raw'][iPt].SetLineWidth(4)
    
    graphs['rebin2'][iPt].SetLineColor(ROOT.kBlue)
    graphs['rebin2'][iPt].SetLineWidth(4)
    graphs['rebin2'][iPt].SetLineStyle(7) # 蓝色虚线
    graphs['rebin2'][iPt].Scale(1/2)
    
    graphs['rebin4'][iPt].SetLineColor(ROOT.kGreen+2)
    graphs['rebin4'][iPt].SetLineWidth(4)
    graphs['rebin4'][iPt].SetLineStyle(9) # 绿色点划线
    graphs['rebin4'][iPt].Scale(1/4)
    
    # 获取三条平滑采样线的极值，自动配置视口，绝对不出界
    max_y = max(graphs['raw'][iPt].GetMaximum(), graphs['rebin2'][iPt].GetMaximum(), graphs['rebin4'][iPt].GetMaximum())
    min_y = min(graphs['raw'][iPt].GetMinimum(), graphs['rebin2'][iPt].GetMinimum(), graphs['rebin4'][iPt].GetMinimum())
    
    # 智能撑开 Y 轴范围
    graphs['raw'][iPt].SetMaximum(max_y + 0.15 * (max_y - min_y))
    graphs['raw'][iPt].SetMinimum(min_y - 0.05 * (max_y - min_y))
    
    # 开始绘图（用 "HIST" 模式把采样点当作平滑线画出来）
    graphs['raw'][iPt].Draw("HIST") 
    graphs['rebin2'][iPt].Draw("HIST Same")
    graphs['rebin4'][iPt].Draw("HIST Same")
    
    # 构建完美的图例
    leg = ROOT.TLegend(0.68, 0.78, 0.93, 0.93)
    legendList.append(leg)
    leg.SetFillStyle(0)
    leg.SetBorderSize(0)
    leg.SetTextSize(0.025)
    
    filename = os.path.basename(root_files['raw'][iPt])
    pt_info = filename.split("fit_")[-1].split("_PtHadBin")[0]
    leg.SetHeader(f"Bin {iPt}: {pt_info}", "C")
    
    leg.AddEntry(graphs['raw'][iPt], "Raw Dataset Fit", "L")
    leg.AddEntry(graphs['rebin2'][iPt], "Rebin 2 Dataset Fit", "L")
    leg.AddEntry(graphs['rebin4'][iPt], "Rebin 4 Dataset Fit", "L")
    leg.Draw()
    
    canvas.Modified()
    canvas.Update()
    
    # 导出
    canvas.SaveAs(f"{outPath}/fit_comparison_{iPt}.png")
    print(f"Successfully generated comparison plot: {outPath}/fit_comparison_{iPt}.png")

print("\n[🎉 SUCCESS] 所有拟合对比图已完美导出！完美绕过了所有 C++ 字符串解析死锁！")

AttributeError: <cppyy.gbl.TROOT object at 0x7cec05e6bc80> object has no attribute 'SetStats'

: 